# Gemma-Titans Hybrid: TPU v5e Online Learning Prototype

This notebook implements a hybrid architecture combining **Gemma-2B** with **Titans Neural Long-Term Memory (NLTM)**.

**Architecture Key Features:**
- **Parallel Memory:** Titans memory runs alongside standard Attention.
- **Learned Gating:** A scalar gate balances short-term context and long-term memory.
- **Modular Weights:** We only train and save the "Titans Delta" (~50MB), keeping the 5GB Gemma weights frozen.

## 1. Environment Setup (TPU v5e-1)
Run the cells below to install the JAX/Flax stack and the official Gemma framework.

In [1]:
#
!pip install tensorboard typeguard==4.4.1 gemma==3.3.0 flax==0.12.5 optax==0.2.6 einx datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 90.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.9/188.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.5/139.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.5/180.5 kB 16.1 MB/

In [1]:
# Clone the repository
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 130 (delta 71), reused 91 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 879.28 KiB | 20.93 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/Titans_jax


## 2. Initialization & Model Surgery
We initialize the hybrid structure and extract the initial "Titans Delta" weights.

In [3]:
import jax
import jax.numpy as jnp
import os
import gc

from gemma import gm
from gemma_titans import Gemma3_1B_Titans
from model_loader import stitch_hybrid_model, load_titans_delta

# 1. Initialize Hybrid Gemma3 1B Model
# Using the pattern from deepmind_finetuning.ipynb: gm.nn.Gemma3_1B
# We created a subclass in gemma_titans.py that uses TitansBlocks
model = Gemma3_1B_Titans(dtype=jnp.bfloat16)

print("Initializing Hybrid Structure (bfloat16)...")
rng = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((1, 1), dtype=jnp.int32)
# JIT compile init to run on TPU and prevent CPU RAM OOM
init_fn = jax.jit(model.init)
params = init_fn(rng, tokens=dummy_tokens)['params']
gc.collect() # Force free RAM
print("Initialization complete. CPU RAM freed.")

import orbax.checkpoint as ocp

# Extract only the Titans parameters (The Delta) from random init
def filter_titans(path, val):
    path_str = "/".join([str(p.key) if hasattr(p, 'key') else str(p.name) if hasattr(p, 'name') else str(p) for p in path])
    if 'memory' in path_str or 'memory_gate' in path_str:
        return val
    return None

titans_delta = jax.tree_util.tree_map_with_path(filter_titans, params)

# Save initial random titans delta
print("Saving initial Titans Delta to ./titans_delta_init...")
checkpointer = ocp.StandardCheckpointer()
if os.path.exists("./titans_delta_init"):
    import shutil
    shutil.rmtree("./titans_delta_init")
checkpointer.save(os.path.abspath("./titans_delta_init"), titans_delta)

print("Loading official Gemma-3 1B weights...")
# Using gm.ckpts.LoadCheckpoint as in deepmind_finetuning.ipynb or direct load_params
base_gemma_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

print("Stitching official Gemma weights with initialized Titans memory...")
params = stitch_hybrid_model(base_gemma_params, titans_delta)
print("Model loaded and stitched successfully!")
gc.collect()

Initializing Hybrid Structure (bfloat16)...
Initialization complete. CPU RAM freed.
Saving initial Titans Delta to ./titans_delta_init...
Loading official Gemma-3 1B weights...


Stitching official Gemma weights with initialized Titans memory...
Model loaded and stitched successfully!


3

## 3. Dataset Preparation
We use a subset of OpenAssistant for dialogue training.

In [4]:
from datasets import load_dataset

dataset = load_dataset("OpenAssistant/oasst1", split="train[:100]")
print(f"Loaded {len(dataset)} dialogue samples.")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

Loaded 100 dialogue samples.


## 4. Phase 1: Continued Pre-Training (CPT)
We freeze the Gemma weights and train only the `memory` and `memory_gate` parameters.

In [ ]:
import optax
from flax.traverse_util import flatten_dict, unflatten_dict

# Define the mask: Only train Titans parameters
def is_trainable(path, val):
    path_str = "/".join([str(p.key) if hasattr(p, 'key') else str(p.name) if hasattr(p, 'name') else str(p) for p in path])
    return 'memory' in path_str or 'memory_gate' in path_str

mask = jax.tree_util.tree_map_with_path(is_trainable, params)

# --- MEMORY OPTIMIZATION: optax.multi_transform ---
# Instead of manually splitting and merging params (which can cause memory spikes
# or require careful handling of FrozenDicts), we use optax.multi_transform.
# We assign trainable parameters to 'adam' and frozen ones to 'set_to_zero'.
# JAX's JIT compiler is smart enough to prune the backward pass for zeroed gradients.

# 1. Create a pytree of labels ('trainable' or 'frozen') matching the params structure
param_partitions = jax.tree_util.tree_map(lambda mask_val: 'trainable' if mask_val else 'frozen', mask)

# 2. Define the optimizer mapping
partition_optimizers = {
    'trainable': optax.adam(learning_rate=1e-4),
    'frozen': optax.set_to_zero()
}

# 3. Create the multi-transform optimizer
tx = optax.multi_transform(partition_optimizers, param_partitions)
opt_state = tx.init(params)
print("Optimizer initialized with optax.multi_transform (frozen params set to zero).")

@jax.jit
def train_step(p, opt_st, tokens):
    def loss_fn(params_inner):
        logits = model.apply({'params': params_inner}, tokens=tokens).logits
        return jnp.mean(logits**2) # Dummy loss for prototype demo

    loss, grads = jax.value_and_grad(loss_fn)(p)
    updates, opt_st = tx.update(grads, opt_st, p)
    new_p = optax.apply_updates(p, updates)
    return new_p, opt_st, loss

print("Starting training step...")
params, opt_state, loss = train_step(params, opt_state, dummy_tokens)
print(f"Initial loss: {loss}")


Optimizer initialized with optax.multi_transform (frozen params set to zero).
Starting training step...


## 5. Dialogue Inference with Dynamic Memory
This demonstrates how the model updates its internal state (`TitansLayerCache`) during a conversation.

In [ ]:
def dialogue_turn(params, cache, user_input_tokens):
    # 1. Forward pass returns updated cache (including updated Titans memory)
    output = model.apply({'params': params}, tokens=user_input_tokens, cache=cache)
    logits = output.logits
    new_cache = output.cache

    # 2. Logic to extract the response token would go here
    return logits, new_cache

# Initial state
cache = model.init_cache(batch_size=1, dtype=jnp.bfloat16, cache_length=512)

print("Simulation: User enters a turn...")
user_tokens = jnp.array([[1, 2, 3, 4]])
logits, cache = dialogue_turn(params, cache, user_tokens)
print("Memory state updated automatically in cache.")

## 6. Save/Load Titans Delta
We save ONLY the trained memory weights to keep the file size minimal.

In [ ]:
import orbax.checkpoint as ocp

def save_titans_only(params, path):
    # Filter to save only Titans parameters
    titans_delta = jax.tree_util.tree_map_with_path(
        lambda path, val: val if is_trainable(path, val) else None,
        params
    )

    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(os.path.abspath(path), titans_delta, force=True)
    print(f"Saved Titans Delta to {path}")

save_titans_only(params, "./titans_delta_init")